# RoBERTa-Large Crypto Classifier (Colab)

Steps:
1. Runtime → Change runtime type → GPU.
2. Upload `crypto_twitter_dataset2.csv` (positive) and `non_crypto_tweets.csv` (negative) to `/content/` or mount Drive.
3. Run cells top to bottom. Model and tokenizer save to `/content/models/crypto-detector-crypto2-roberta-large`. Optional CV saves to `/content/models/crypto-detector-crypto2-roberta-large-cv`.

Data expectations:
- Positive CSV: column `tweet_text` or `clean_text` or `full_text`.
- Negative CSV: column `full_text`.
- Separator is `;`.

Notes:
- Train/test sizes adapt automatically if данных меньше, чем целевые 3000/700 на класс.
- CV (Stratified K-fold) можно отключить `DO_CV=False`.
- Для Colab используется только ресурсы Google (GPU/CPU/RAM); ваш компьютер не нагружается, кроме загрузки/выгрузки файлов.

In [ ]:
# Install dependencies (Colab)
!pip -q install transformers datasets scikit-learn matplotlib

In [ ]:
# Imports
import os
import random
import re
import html
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
import matplotlib.pyplot as plt

print("CUDA available:", torch.cuda.is_available())


In [ ]:
# Config (multi-model sequential fine-tuning)
POS_PATH = "/content/trashed_crypto_dataset.csv"   # очищенный позитивный класс
NEG_PATH = "/content/non_crypto_tweets.csv"
OUT_ROOT = "/content/models/crypto-detector-multi"  # корневая папка для всех моделей

# Базовые настройки
EPOCHS = 5
BATCH_SIZE = 4                     # для крупных моделей; эффективный batch 16 при GRAD_ACCUM_STEPS=4
GRAD_ACCUM_STEPS = 4
MAX_LEN = 192
SEED = 42
LEARNING_RATE = 1.5e-5
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.1
EARLY_STOPPING_PATIENCE = 2
FP16 = True
LABEL_SMOOTHING = 0.05
SCHEDULER_TYPE = "cosine"
GRADIENT_CHECKPOINTING = False      # отключено из-за повторного backward
DO_CV = False

# Три модели: текущая + две мощнее
MODELS = [
    {"name": "microsoft/deberta-v3-large", "lr": 1.5e-5, "alias": "deberta-v3-large"},
    {"name": "microsoft/deberta-v3-xxlarge", "lr": 1.2e-5, "alias": "deberta-v3-xxlarge"},
    {"name": "microsoft/deberta-v2-xxlarge", "lr": 1.2e-5, "alias": "deberta-v2-xxlarge"},
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# Helper functions

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    return text


def load_positive(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep=None, engine="python")
    except Exception:
        df = pd.read_csv(path, sep=";")
    text_col = None
    for cand in ["text", "tweet_text", "clean_text", "full_text"]:
        if cand in df.columns:
            text_col = cand
            break
    if text_col is None:
        raise ValueError("No text column found in positive dataset")
    df = df[[text_col]].rename(columns={text_col: "text"})
    df = df.dropna(subset=["text"])
    df["text"] = df["text"].apply(clean_text)
    df = df[df["text"].ne("")]
    df["label"] = 1
    df["label"] = df["label"].astype(int)
    return df


def load_negative(path: str, total_needed: int) -> pd.DataFrame:
    try:
        df_raw = pd.read_csv(path, sep=None, engine="python")
    except Exception:
        df_raw = pd.read_csv(path, sep=";")
    if "full_text" not in df_raw.columns:
        raise ValueError("full_text column missing in negative dataset")
    df_full = df_raw[["full_text"]].rename(columns={"full_text": "text"})
    df_full = df_full.dropna(subset=["text"])
    df_full["text"] = df_full["text"].apply(clean_text)
    df_full = df_full[df_full["text"].ne("")]
    df_full["label"] = 0
    df_full["label"] = df_full["label"].astype(int)
    return df_full.copy()


def make_splits(pos: pd.DataFrame, neg: pd.DataFrame) -> DatasetDict:
    min_len = min(len(pos), len(neg))
    test_size = max(int(0.1 * min_len), 1)
    val_size = max(int(0.1 * min_len), 1)
    train_size = max(min_len - test_size - val_size, 1)

    pos_sample = pos.sample(min_len, random_state=SEED).reset_index(drop=True)
    neg_sample = neg.sample(min_len, random_state=SEED).reset_index(drop=True)

    pos_train = pos_sample.iloc[:train_size]
    pos_val = pos_sample.iloc[train_size:train_size + val_size]
    pos_test = pos_sample.iloc[train_size + val_size:train_size + val_size + test_size]

    neg_train = neg_sample.iloc[:train_size]
    neg_val = neg_sample.iloc[train_size:train_size + val_size]
    neg_test = neg_sample.iloc[train_size + val_size:train_size + val_size + test_size]

    train_df = pd.concat([pos_train, neg_train], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    val_df = pd.concat([pos_val, neg_val], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_df = pd.concat([pos_test, neg_test], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

    for df in (train_df, val_df, test_df):
        df["label"] = df["label"].astype(int)

    print(f"Using train={len(train_df)}, val={len(val_df)}, test={len(test_df)} total")
    ds = DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "val": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True)),
    })
    return ds


def tokenize_dataset(ds: DatasetDict, tokenizer: AutoTokenizer) -> DatasetDict:
    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN, padding=False)

    cols_to_remove = [c for c in ds["train"].column_names if c not in ["text", "label"]]
    return ds.map(tok, batched=True, remove_columns=cols_to_remove)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = (preds == labels).mean()
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
# Normalize datasets (optional) → creates text,label CSVs for inspection
try:
    pos_raw = pd.read_csv(POS_PATH, sep=None, engine="python")
except Exception:
    pos_raw = pd.read_csv(POS_PATH, sep=";")
try:
    neg_raw = pd.read_csv(NEG_PATH, sep=None, engine="python")
except Exception:
    neg_raw = pd.read_csv(NEG_PATH, sep=";")

# Pick text columns
pos_text_col = next((c for c in ["text", "tweet_text", "clean_text", "full_text"] if c in pos_raw.columns), None)
neg_text_col = "full_text" if "full_text" in neg_raw.columns else None
if pos_text_col is None or neg_text_col is None:
    raise ValueError("Required text columns not found to normalize datasets")

pos_clean = pos_raw[[pos_text_col]].rename(columns={pos_text_col: "text"}).dropna()
neg_clean = neg_raw[[neg_text_col]].rename(columns={neg_text_col: "text"}).dropna()

for df in (pos_clean, neg_clean):
    df["text"] = df["text"].astype(str).str.strip()
    df.drop(df[df["text"] == ""].index, inplace=True)

pos_clean["label"] = 1
neg_clean["label"] = 0

pos_out = "/content/crypto_pos_normalized.csv"
neg_out = "/content/crypto_neg_normalized.csv"
pos_clean.to_csv(pos_out, index=False)
neg_clean.to_csv(neg_out, index=False)
print(f"Saved normalized files: {pos_out}, {neg_out}")


In [ ]:
# Train three models sequentially on the same data
pos = load_positive(POS_PATH)
neg = load_negative(NEG_PATH, 0)
print(f"Pos: {len(pos)}, Neg: {len(neg)}")
print('Positive sample:', pos.head(2).to_dict())
print('Negative sample:', neg.head(2).to_dict())

# Shared splits (balanced by min class size)
ds = make_splits(pos, neg)

results = {}
best_model_name = None
best_model_dir = None
best_f1 = -1.0

for cfg in MODELS:
    model_name = cfg["name"]
    lr = cfg.get("lr", LEARNING_RATE)
    alias = cfg.get("alias", model_name.split("/")[-1])
    out_dir = os.path.join(OUT_ROOT, alias)
    os.makedirs(out_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    ds_tok = tokenize_dataset(ds, tokenizer)
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    # Disable cache to be safe; gradient checkpointing выключен в конфиге
    model.config.use_cache = False

    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",  # validation each epoch
        save_strategy="epoch",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        fp16=FP16,
        learning_rate=lr,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1",
        greater_is_better=True,
        save_total_limit=1,
        report_to=[],
        label_smoothing_factor=LABEL_SMOOTHING,
        lr_scheduler_type=SCHEDULER_TYPE,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    print(f"\n===== Training {model_name} (alias {alias}) =====")
    train_result = trainer.train()
    val_metrics = trainer.evaluate()
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    print("Validation metrics:", val_metrics)
    print("Test metrics:", test_metrics)

    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    results[alias] = {"val": val_metrics, "test": test_metrics, "log_history": trainer.state.log_history}

    f1_current = test_metrics.get("eval_f1", val_metrics.get("eval_f1", 0))
    if f1_current > best_f1:
        best_f1 = f1_current
        best_model_name = alias
        best_model_dir = out_dir

print("\nCompleted training of all models.")
print("Best model:", best_model_name, "with F1=", best_f1)


In [ ]:
# (Deprecated duplicate) — основная тренировка вынесена в предыдущую ячейку.

In [ ]:
# Basic tests
assert len(ds["train"]) > 0 and len(ds["test"]) > 0
sample = AutoTokenizer.from_pretrained(MODELS[0]["name"]).__call__("short text", truncation=True, max_length=MAX_LEN)
print("Basic tests passed.")

In [ ]:
# Inference helper (uses лучшую модель по тестовому F1)
from transformers import TextClassificationPipeline

if best_model_dir is None:
    best_model_dir = os.path.join(OUT_ROOT, MODELS[0].get("alias", MODELS[0]["name"].split("/")[-1]))

best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
best_tokenizer = AutoTokenizer.from_pretrained(best_model_dir)

pipe = TextClassificationPipeline(
    model=best_model,
    tokenizer=best_tokenizer,
    return_all_scores=True,
    device=0 if torch.cuda.is_available() else -1,
)

example = "Bitcoin rally is breaking all records"
out = pipe(example)
print(example)
print(out)
